In [93]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, kstwobign, norm

In [94]:
X = np.array([98.7, 101.4, 102.7, 103.6, 104.2, 105.7, 105.9, 112.6, 121.5, 123.5])
Y = np.array([108.9, 110.3, 110.9, 113.6, 116.7])
print(X, Y)

[ 98.7 101.4 102.7 103.6 104.2 105.7 105.9 112.6 121.5 123.5] [108.9 110.3 110.9 113.6 116.7]


In [95]:
n, m = len(X), len(Y)
combined = np.concatenate([X, Y])
labels = np.array([0]*n + [1]*m)   # 0 – X, 1 – Y
idx = np.argsort(combined)
combined_sorted = combined[idx]
labels_sorted = labels[idx]

i, j = 0, 0          # текущие счётчики
max_diff = 0.0
max_diff_indexes = None, None
unique_vals = np.unique(combined_sorted)

for val in unique_vals:
    mask = (combined_sorted == val)
    countX = np.sum(labels_sorted[mask] == 0)
    countY = np.sum(labels_sorted[mask] == 1)

    diff = abs(i/n - j/m)
    if diff > max_diff:
        max_diff = diff

    i += countX
    j += countY
    diff = abs(i/n - j/m)
    if diff > max_diff:
        max_diff_indexes = i, j, i/n, j/m
        max_diff = diff

print(f"KS statistic (manual) = {max_diff}")
print(max_diff_indexes)
D = max_diff

KS statistic (manual) = 0.7
(np.int64(7), np.int64(0), np.float64(0.7), np.float64(0.0))


In [96]:
# ρ(X) = D * sqrt(n*m/(n+m))
rho = D * np.sqrt(n * m / (n + m))

alpha = 0.01
c = kstwobign.ppf(1 - alpha)

print(f"sup  (D_n) = {D:.6f}")
print(f"ρ(X)        = {rho:.6f}")
print(f"c (α=0.01)  = {c:.6f}")
print("не отвергаем H0" if abs(rho) < c else "отвергаем H0")

sup  (D_n) = 0.700000
ρ(X)        = 1.278019
c (α=0.01)  = 1.627624
не отвергаем H0


In [97]:
def kolmogorov_sf(rho, max_k=100):
    """Survival function (1 - CDF) for Kolmogorov distribution"""
    if rho <= 0:
        return 1.0
    s = 0.0
    for k in range(1, max_k + 1):
        term = (-1)**(k-1) * np.exp(-2 * k**2 * rho**2)
        s += term
        if abs(term) < 1e-15:
            break
    return 2 * s

p = kolmogorov_sf(rho)
print(f"p-value (manual series) = {p:.6f}")

print("не отвергаем H0" if p >= 0.05 else "отвергаем H0")

p-value (manual series) = 0.076262
не отвергаем H0


In [98]:
def mann_whitney_u_test(sample1, sample2, alpha=0.05):
    """
    Проверка гипотезы H0: распределения двух выборок одинаковы.
    Используется критерий Манна-Уитни с нормальной аппроксимацией.
    """
    n1 = len(sample1)
    n2 = len(sample2)

    # 1. Объединяем выборки с метками групп
    combined = [(value, 1) for value in sample1] + [(value, 2) for value in sample2]

    # 2. Сортируем по значению
    combined.sort(key=lambda x: x[0])

    # 3. Присваиваем ранги (средний ранг для одинаковых значений)
    ranks = [0] * len(combined)
    i = 0
    while i < len(combined):
        j = i
        # Находим группу одинаковых значений
        while j < len(combined) and combined[j][0] == combined[i][0]:
            j += 1
        # Количество одинаковых значений
        k = j - i
        # Средний ранг для этой группы
        avg_rank = (i + 1 + j) / 2.0
        for t in range(i, j):
            ranks[t] = avg_rank
        i = j

    # 4. Сумма рангов для первой выборки
    R1 = 0
    for idx, (_, group) in enumerate(combined):
        if group == 1:
            R1 += ranks[idx]

    # 5. Вычисляем U-статистику
    U1 = n1 * n2 + n1 * (n1 - 1) / 2.0 - R1
    U2 = n1 * n2 - U1  # эквивалентно по формуле
    print("U", U1, U2)
    U = min(U1, U2)

    # 6. Математическое ожидание и дисперсия
    EU = n1 * n2 / 2.0
    DU = n1 * n2 * (n1 + n2 + 1) / 12.0

    # 7. Z-статистика
    Z = (U - EU) / math.sqrt(DU)

    # 8. p-value
    p_value = 1 - math.erf(abs(Z) / math.sqrt(2))

    # 9. C
    C = norm.ppf(1 - 0.05 / 2)
    print("C", C)

    print(f"Z {Z:}")
    print(f"p-value = {p_value:.6f}")
    print("не отвергаем H0" if abs(Z) < C else "отвергаем H0")

mann_whitney_u_test(X, Y)

U 27.0 23.0
C 1.959963984540054
Z -0.2449489742783178
p-value = 0.806496
не отвергаем H0
